# QPINO exploration notebook
Loads the saved results of the experiments (run `pytest`, then the scripts in `experiments/`) and reproduces a few quick views. Synthetic Fisher-KPP validation is kept separate from the exploratory JHU COVID-19 fit.

In [ ]:
import sys, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '..')
from src import galerkin as gk
M = lambda f: json.load(open(f'../results/metrics/{f}'))

## 1. M=2 Galerkin system: explicit equations vs direct spectral projection

In [ ]:
a = np.random.default_rng(0).normal(size=(5, 5))
print(np.abs(gk.galerkin_rhs(a, 0.1, 1.0, 1.0) - gk.galerkin_rhs_by_projection(a, 0.1, 1.0, 1.0)).max())
print('coupling Hessian (structured ZZ graph):')
print((gk.coupling_hessian() > 0).astype(int))

## 2. Synthetic validation summary

In [ ]:
S = M('synthetic_results.json')
pd.DataFrame({k: {m: v[m]['mean'] for m in ['rel_l2_increment', 'rollout_rel_l2', 'pde_residual_rms_trapezoid']} | {'params': v['n_params']} for k, v in S['summary'].items()}).T

## 3. JHU COVID-19 I-10 transect (exploratory)

In [ ]:
Z = np.load('../data/processed/covid_i10_field.npz')
plt.imshow(Z['u'].T, aspect='auto', origin='lower', cmap='magma'); plt.xlabel('day since 2020-03-01'); plt.ylabel('bin along I-10'); plt.colorbar(label='u')

In [ ]:
Cr = M('covid_results.json')
pd.DataFrame({k: {m: v[m]['mean'] for m in ['rel_l2_increment', 'field_rmse_per100k_onestep', 'rollout_rel_l2', 'field_rmse_per100k_rollout'] if m in v} for k, v in Cr['summary'].items()}).T

## 4. Figures
All figures are in `results/figures/` (fig01-fig12).